In [ ]:
import pandas as pd

df = pd.read_parquet(
    "../data/processed/orders_clean.parquet"
)

print(df.shape)

In [ ]:
retailer_features = (
    df.groupby("customerId")
    .agg(
        total_orders=(
            "orderNumber",
            "nunique"
        ),

        total_qty=(
            "effective_qty",
            "sum"
        ),

        unique_products=(
            "skuNumber",
            "nunique"
        ),

        last_order=(
            "createdAt",
            "max"
        ),

        first_order=(
            "createdAt",
            "min"
        ),

        shop_type=(
            "shopType",
            "first"
        ),

        retailer_type=(
            "retailerType",
            "first"
        ),

        hub_name=(
            "hubName",
            "first"
        )
    )
    .reset_index()
)

retailer_features.head()

In [ ]:
TODAY = df["createdAt"].max()

retailer_features[
    "days_since_last_order"
] = (
    TODAY -
    retailer_features["last_order"]
).dt.days

retailer_features.head()

In [ ]:
fav_category = (
    df.groupby("customerId")
    ["category"]
    .agg(
        lambda x:
        x.value_counts().index[0]
    )
    .reset_index()
)

fav_category.columns = [
    "customerId",
    "favorite_category"
]

fav_category.head()

In [ ]:
retailer_features = (
    retailer_features.merge(
        fav_category,
        on="customerId",
        how="left"
    )
)

retailer_features.head()

In [ ]:
fav_brand = (
    df.groupby("customerId")
    ["brand"]
    .agg(
        lambda x:
        x.value_counts().index[0]
    )
    .reset_index()
)

fav_brand.columns = [
    "customerId",
    "favorite_brand"
]

In [ ]:
retailer_features = (
    retailer_features.merge(
        fav_brand,
        on="customerId",
        how="left"
    )
)

retailer_features.head()

In [ ]:
retailer_features[
    "spend_segment"
] = pd.qcut(
    retailer_features[
        "total_qty"
    ],
    q=3,
    labels=[
        "Low",
        "Medium",
        "High"
    ]
)

retailer_features[
    [
        "customerId",
        "total_qty",
        "spend_segment"
    ]
].head()

In [ ]:
retailer_features.to_parquet(
    "../data/processed/retailer_features.parquet",
    index=False
)

print("Saved")

In [ ]:
retailer_features[
    "spend_segment"
].value_counts()

In [ ]:
retailer_features.head()

In [ ]:
retailer_features["spend_segment"].value_counts()

In [ ]:
retailer_features.shape

In [ ]:
import pandas as pd

retailer_features = pd.read_parquet(
    "../data/processed/retailer_features.parquet"
)

print(retailer_features.shape)

In [ ]:
from pathlib import Path

print(Path("../data/processed/retailer_features.parquet").resolve())

In [ ]:
print(retailer_features.shape)
print(retailer_features["customerId"].nunique())